# 7 · Classification — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch07_classification/05_solutions.ipynb)

Les réponses des exercices, exécutées. Les démarches détaillées (le *pourquoi*, les erreurs fréquentes, les variantes) sont dans `05_solutions.md`. Les cellules marquées `answer` enregistrent les réponses vérifiées par `wb.check` (`tools/build_answers.py`).

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 7.Q1 à 7.Q11, 7.R1 à 7.R3, 7.1 à 7.6 | vérifier tes réponses courtes | 🧠 🔁 ✏️ ∂ | ★ à ★★★ | |
| A | 7.11 à 7.16 | frontières, distances, centroïde le plus proche, carte de probabilité, un-contre-tous avec des centroïdes (*prochaine session de génération*) | | | |
| B | 7.17 à 7.21 | k-means, DBSCAN et HDBSCAN avec scikit-learn, puis la grande dimension (*prochaine session de génération*) | | | |
| C | 7.22 à 7.24 | un-contre-tous et un-contre-un génériques dans mylearn.multiclass (*prochaine session de génération*) | | | |
| D | 7.25 à 7.31 | k-means de zéro dans mylearn.cluster, silhouette, choix de k, phénomène de Hughes, défi (*prochaine session de génération*) | | | |

In [1]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        if pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("ref")  # reference implementation

🧪 Deep Learning Workbook : environnement
   Plateforme : Local (Linux x86_64, 2 CPU)
   Python 3.13.13 | numpy 2.1.3 | pandas 2.2.3 | matplotlib 3.10.0 | scikit-learn 1.6.1 | scipy 1.16.3 | torch 2.11.0+cpu | torchvision 0.26.0+cpu
   Device : cpu | Graine : 42 | FAST_MODE : True


## Objectifs et rappel express

- Lire des régions et des frontières de décision, et régler un seuil d'après le coût des erreurs.
- Traiter plusieurs classes avec des classifieurs binaires : un-contre-tous et un-contre-un.
- Programmer le centroïde le plus proche et k-means (Lloyd, k-means++), et choisir $k$ avec l'inertie et la silhouette ; savoir quand préférer DBSCAN ou HDBSCAN.
- Mesurer ce que la dimension fait aux densités et aux distances.

**Rappel express.** $N_{\text{OvR}} = K$, $N_{\text{OvO}} = \frac{K(K-1)}{2}$ ; seuil de coût minimal $t^* = \frac{C_{FP}}{C_{FP} + C_{FN}}$ ; $\lVert \mathbf{a} - \mathbf{b} \rVert^2 = \lVert \mathbf{a} \rVert^2 - 2\,\mathbf{a} \cdot \mathbf{b} + \lVert \mathbf{b} \rVert^2$ ; inertie $J = \sum_i \lVert \mathbf{x}_i - \boldsymbol{\mu}_{c_i} \rVert^2$ ; silhouette $s = \frac{b - a}{\max(a, b)}$ ; densité $\rho = \frac{n}{b^d}$ ; peau d'une boule $1 - (1 - \varepsilon)^d$. En Python : `sklearn.cluster.KMeans`, `DBSCAN`, `HDBSCAN`, `sklearn.metrics.silhouette_score`, `sklearn.multiclass`, `rng.choice(n, p=w / w.sum())`.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ 7.1 à 7.4, ∂ 7.5 et 7.6)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.375` (en Python, le séparateur décimal est un **point** ; `0,375` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"D"`) ; plusieurs nombres : une liste (`[2, 5]`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », le quiz Q10, ∂ 7.7, la réflexion et l'entretien se corrigent avec `05_solutions.md`.

In [2]:
# The short answers only use the numbers of their statements (02_exercices.md)
import math

import numpy as np


def entropy_bits(p):
    """Entropy in bits of a distribution (counts or proportions), with 0 log 0 = 0."""
    p = np.asarray(p, dtype=float)
    p = p[p > 0] / p.sum()
    return float(-(p * np.log2(p)).sum())


def posterior(prior, f1, f0):
    """Bayes rule for two classes: P(class 1 | x) from the prior and the two densities at x."""
    return prior * f1 / (prior * f1 + (1 - prior) * f0)


def n_ovo(k):
    """Number of one-versus-one duels for k classes."""
    return k * (k - 1) // 2


A_R3, B_R3 = np.array([3, -1, 2]), np.array([1, 2, 0])            # 7.R3

# 7.2: the six duels in the order A-B, A-C, A-D, B-C, B-D, C-D and the index of each winner
WINNERS_72A = [1, 0, 3, 1, 1, 3]                                    # first point
WINNERS_72B = [0, 2, 0, 1, 3, 2]                                    # second point


def votes(winners):
    counts = np.zeros(4, dtype=int)
    for winner in winners:
        counts[winner] += 1
    return counts


# 7.3: Lloyd's algorithm by hand, k = 2, starting centres P1 and P3
P_73 = np.array([[1, 1], [2, 1], [1, 3], [5, 4], [6, 5], [7, 4]], dtype=float)


def assign(points, centres):
    return ((points[:, None, :] - centres[None, :, :]) ** 2).sum(axis=2).argmin(axis=1)


def update(points, labels, k):
    return np.array([points[labels == j].mean(axis=0) for j in range(k)])


def inertia(points, centres, labels):
    return float(((points - centres[labels]) ** 2).sum())


C0_73 = P_73[[0, 2]]
L1_73 = assign(P_73, C0_73)
C1_73 = update(P_73, L1_73, 2)
L2_73 = assign(P_73, C1_73)
C2_73 = update(P_73, L2_73, 2)
L3_73 = assign(P_73, C2_73)

# 7.4: densities n / b**d
N_74, B_74 = 360, 6
EMPTY_74 = (124 / 125) ** 10                                        # the book's 10 eggs in 125 small cubes

# 7.6: volume of the unit ball, V_d = 2 pi / d V_(d-2)
V = {1: 2.0, 2: math.pi}
for d in range(3, 101):
    V[d] = 2 * math.pi / d * V[d - 2]

**7.Q1 — Label, prédiction, vérité terrain : le vocabulaire**

In [3]:
wb.record("7.Q1a", "C", mistakes={"le label (étiquette) est la classe fixée par l'expert": "A",
          "la vérité terrain est la classe que l'on tient pour juste, celle de l'expert": "B",
          "le livre donne la valeur réelle (actual value) comme un autre nom du label": "D"})
wb.record("7.Q1b", True, mistakes={"le mirage est un savoir-faire humain : un mireur ne peut-il jamais se tromper sur un œuf ambigu ?": False})
wb.record("7.Q1c", False, mistakes={"ce qui compte est la réussite sur des œufs que le modèle n'a jamais vus (ch. 1)": True})

WB_ANSWER {"hash": "d935b4ffcd9143d9cd40a822c1ba33f4aeda1235894297225802891ec3736cc6", "id": "7.Q1a", "kind": "str", "mistakes": {"4ce76d5031ee3b2937c74aa705abec83dd0cb5b279ec05ba26f65121bdd723a9": "le livre donne la valeur réelle (actual value) comme un autre nom du label", "4ef79991ac627b1d82e15d6d7cf11b87ca8ff1a6c7427be055494238ed91202c": "la vérité terrain est la classe que l'on tient pour juste, celle de l'expert", "d4f5bcbe9acb78241bbea31d7cd2a28cf3b2a82d642e85c7584551629a77dbd6": "le label (étiquette) est la classe fixée par l'expert"}}
📝 Ex 7.Q1a : réponse enregistrée (str).
WB_ANSWER {"hash": "d478b6ee1560c730a6a68cb690040ea80f108a2d2a96ade93e42eb33aa73320c", "id": "7.Q1b", "kind": "bool", "mistakes": {"5cc46a4b367ca01fc35fe7c1428578f90fb0623aea7d220b9da58d9210798c80": "le mirage est un savoir-faire humain : un mireur ne peut-il jamais se tromper sur un œuf ambigu ?"}}
📝 Ex 7.Q1b : réponse enregistrée (bool).
WB_ANSWER {"hash": "4dc52254afed2cfc4028ba97143a14884b9da19f6901f1ff

**7.Q2 — Binaire, multi-classe ou multi-étiquette ? Cinq situations**

In [4]:
wb.record("7.Q2a", "B", mistakes={"spam ou non : il n'y a que deux classes": "C",
          "un e-mail ne reçoit qu'une seule réponse parmi deux": "E"})
wb.record("7.Q2b", "C", mistakes={"trois espèces possibles : plus de deux classes": "B",
          "un manchot n'a qu'une espèce : une seule classe par exemple": "E"})
wb.record("7.Q2c", "E", mistakes={"une photo peut recevoir plusieurs mots-clés à la fois": "C",
          "il y a bien plus de deux mots-clés possibles, et une photo peut en avoir plusieurs": "B"})
wb.record("7.Q2d", "C", mistakes={"dix chiffres possibles : plus de deux classes": "B",
          "une image ne montre qu'un seul chiffre : une seule classe par exemple": "E"})
wb.record("7.Q2e", "B", mistakes={"fécondé ou non : il n'y a que deux classes": "C",
          "un œuf est fécondé ou ne l'est pas : une seule réponse parmi deux": "E"})

WB_ANSWER {"hash": "ff5a6f7efb5ab25bea5f8a8687c99e66d20b7aa60c8b29037ec0cc4530e339a0", "id": "7.Q2a", "kind": "str", "mistakes": {"28430519e5ba0f690d68482ddbaf5a828c42497cb701b2b1199196476db36080": "spam ou non : il n'y a que deux classes", "e776ec23db6e2134c80c097e5382dd244fcebc76d84d314fc3b25a65b53ab613": "un e-mail ne reçoit qu'une seule réponse parmi deux"}}
📝 Ex 7.Q2a : réponse enregistrée (str).
WB_ANSWER {"hash": "30a27bec0b654be256093804613529f460d0a94c51354de35c82e937a085d616", "id": "7.Q2b", "kind": "str", "mistakes": {"0bb78474b690d6a6c47168c370820a0c585f7b08eaa592e1aed805e9066b1529": "un manchot n'a qu'une espèce : une seule classe par exemple", "4c17868468537037c63fe09c61859ffb1299a9eae71c1ef85dc80a09f7e0f07a": "trois espèces possibles : plus de deux classes"}}
📝 Ex 7.Q2b : réponse enregistrée (str).
WB_ANSWER {"hash": "6693e82dc6661f2f6fc63fc06cbffe61ebd589e90a804a2cadb06d070206fca1", "id": "7.Q2c", "kind": "str", "mistakes": {"a91bcf0fb7ca8a57172f7b1286bfce2f91589eaab43c

**7.Q3 — Régions et frontières de décision**

In [5]:
wb.record("7.Q3a", False, mistakes={"regarde la figure 7.3 du livre, ou le panneau (b) de la figure des trois situations de la fiche": True})
wb.record("7.Q3b", 3, mistakes={"chaque classe que le classifieur peut prédire occupe au moins un morceau du plan": 2})
wb.record("7.Q3c", True, mistakes={"rien n'oblige une classe à n'occuper qu'un seul morceau : imagine deux paquets d'œufs viables séparés par des quitters": False})
wb.record("7.Q3d", False, mistakes={"un classifieur rend une seule classe par point, même sur la frontière : une convention tranche": True})
wb.record("7.Q3e", False, mistakes={"fais tourner très légèrement la droite entre deux nuages bien séparés : sépare-t-elle encore les deux classes ?": True})

WB_ANSWER {"hash": "bd525834c8ef8aca28271d9c33762b15d1f3c1c66ceb8e5e40c73625f796dd01", "id": "7.Q3a", "kind": "bool", "mistakes": {"293443f723f0a3b18a5236b9c259b72ee7832aa39d98191d368c69d115cd62ce": "regarde la figure 7.3 du livre, ou le panneau (b) de la figure des trois situations de la fiche"}}
📝 Ex 7.Q3a : réponse enregistrée (bool).
WB_ANSWER {"hash": "ba7dfabd87ef91eaa17047f7d45213fe5e24f62c39e0871a6d34566673730cc6", "id": "7.Q3b", "kind": "int", "magnitude": 0, "mistakes": {"d862d063b49b84cacbbbcd7fae6235a5c62880e3d353372faccd63d71bbf1f9a": "chaque classe que le classifieur peut prédire occupe au moins un morceau du plan"}, "sign": 1}
📝 Ex 7.Q3b : réponse enregistrée (int).
WB_ANSWER {"hash": "552688f521f7c54def9363e28107e582e0d38d5cc032c5aca54b4214fe6c439d", "id": "7.Q3c", "kind": "bool", "mistakes": {"ef255289285774e86f68e0edd7fafa226473d031bcfa4dba592fe449762c5cbb": "rien n'oblige une classe à n'occuper qu'un seul morceau : imagine deux paquets d'œufs viables séparés par des 

**7.Q4 — Classes qui se recouvrent : probabilités et politique de seuil**

In [6]:
wb.record("7.Q4a", True, mistakes={"tout point où P ≥ 0,5 vérifie aussi P ≥ 0,2 : un œuf fécondé déjà repéré peut-il être perdu ?": False})
wb.record("7.Q4b", True, mistakes={"les œufs déclarés fécondés au seuil 0,5 le restent au seuil 0,2, qu'ils le soient vraiment ou non : un faux positif peut-il disparaître ?": False})
wb.record("7.Q4c", "A", mistakes={"compare les ensembles {P ≥ 0,5} et {P ≥ 0,2} : l'un contient l'autre": "B",
          "tout point de la région au seuil 0,5 y reste au seuil 0,2 : elle ne peut pas rétrécir": "C"})
wb.record("7.Q4d", 1 / (1 + 7), decimals=3, mistakes={"le seuil est C_FP / (C_FP + C_FN) : il BAISSE quand les faux négatifs coûtent cher": 7 / (1 + 7),
          "le dénominateur est la SOMME des deux coûts, pas l'un des deux": 1 / 7,
          "0,5 ne convient que si les deux erreurs coûtent autant : calcule C_FP / (C_FP + C_FN)": 0.5,
          "0,2 est le seuil choisi au début de l'énoncé : calcule celui qui minimise le coût moyen": 0.2})

WB_ANSWER {"hash": "e8a53fe8dafac63624f20b17dd631631655bc2e44c8302124c45a7ee0f5c6513", "id": "7.Q4a", "kind": "bool", "mistakes": {"48dfddcd715e2d1e236cd7bddd35cbda129e6932726e630751a4ded55a8e87a4": "tout point où P ≥ 0,5 vérifie aussi P ≥ 0,2 : un œuf fécondé déjà repéré peut-il être perdu ?"}}
📝 Ex 7.Q4a : réponse enregistrée (bool).
WB_ANSWER {"hash": "82092022132ac4ef0204a08c00d0a6476622e639f17f8d7eb2f58c71e2f7a979", "id": "7.Q4b", "kind": "bool", "mistakes": {"cb965ab8952a03a58a2985c580a8b57eb8e0f47ec14ca5cb4396c224bae17097": "les œufs déclarés fécondés au seuil 0,5 le restent au seuil 0,2, qu'ils le soient vraiment ou non : un faux positif peut-il disparaître ?"}}
📝 Ex 7.Q4b : réponse enregistrée (bool).
WB_ANSWER {"hash": "d234ea8c9ff76751055fc0a5877ddbf9e36127fb3084b70d44f65583eeb5d878", "id": "7.Q4c", "kind": "str", "mistakes": {"3b080f74d00cef4a59f0c28095500b43c913cfebb8baf7e8ee972d374b2610fd": "compare les ensembles {P ≥ 0,5} et {P ≥ 0,2} : l'un contient l'autre", "cb6ffdd18

**7.Q5 — Cinq mesures par œuf : ce que change (ou non) la dimension**

In [7]:
wb.record("7.Q5a", 5, mistakes={"une dimension par feature : la classe n'est pas une feature": 6})
wb.record("7.Q5b", True, mistakes={"une distance euclidienne est la racine de la somme des carrés des écarts, quel que soit le nombre de coordonnées": False})
wb.record("7.Q5c", False, mistakes={"chaque feature ajoute un terme à chaque distance et à chaque moyenne : le calcul peut-il coûter autant ?": True})
wb.record("7.Q5d", 4, mistakes={"une frontière a une dimension de MOINS que l'espace (une droite dans le plan)": 5})

WB_ANSWER {"hash": "11985c5bdc4f3def1f7951ff7a64c96d2d68ed1463b0a2b128cf992464fe9b42", "id": "7.Q5a", "kind": "int", "magnitude": 0, "mistakes": {"4bc0bc6704cbf90139ea00c93eac339ff2a7668d8cbe18edc8fe823d5e2d0c54": "une dimension par feature : la classe n'est pas une feature"}, "sign": 1}
📝 Ex 7.Q5a : réponse enregistrée (int).
WB_ANSWER {"hash": "8a8386b35a59fcde23e898381f4c06917b257c23f7c3518119dae4de9f80f19c", "id": "7.Q5b", "kind": "bool", "mistakes": {"040148a890b4e2dbdfd19941b039b26fd71f6dcb5e5a4e53497e47d534793ffd": "une distance euclidienne est la racine de la somme des carrés des écarts, quel que soit le nombre de coordonnées"}}
📝 Ex 7.Q5b : réponse enregistrée (bool).
WB_ANSWER {"hash": "d1233d9bf8ff9a5fa9b2b50fbf539a039eefefd37b74ed7910df01320bd0a9a4", "id": "7.Q5c", "kind": "bool", "mistakes": {"bfb271b6a90e20dd1de8514feef3d763a49305b99feaa4f726ec9912960b7311": "chaque feature ajoute un terme à chaque distance et à chaque moyenne : le calcul peut-il coûter autant ?"}}
📝 Ex 7

**7.Q6 — Un-contre-tous : combien de modèles, quelle décision ?**

In [8]:
wb.record("7.Q6a", 7, mistakes={"un-contre-tous : un classifieur par CLASSE, pas un par paire": 21})
wb.record("7.Q6b", "B", mistakes={"compare 0,55 et 0,52": "C", "on prédit la classe de plus GRAND score": "D",
          "0,30 n'est pas le plus grand score": "A"})
wb.record("7.Q6c", False, mistakes={"additionne les quatre probabilités de b)": True})
wb.record("7.Q6d", "B", mistakes={"le plus grand de quatre nombres négatifs est le plus proche de 0": "C",
          "compare −1,2 et −0,4 : lequel est le plus grand ?": "A",
          "compare −0,9 et −0,4 : lequel est le plus grand ?": "D"})

WB_ANSWER {"hash": "e51f060c8e2152670004d0bc2c095e0b05efd69a2790291594ad65c8da353ebe", "id": "7.Q6a", "kind": "int", "magnitude": 0, "mistakes": {"c28eb1d7f2f84c816aabb8563f6dbc92ac52f94f16a4710e6c9a5cb2c3169aee": "un-contre-tous : un classifieur par CLASSE, pas un par paire"}, "sign": 1}
📝 Ex 7.Q6a : réponse enregistrée (int).
WB_ANSWER {"hash": "256d3bf431924a3a3fff7d5abfa660ed59902d3b658e87652cfff017303ed11b", "id": "7.Q6b", "kind": "str", "mistakes": {"36b6d688853b5d7e0fd764e0425598eb47cc5e4876add5491f6fed82eb05ce4e": "0,30 n'est pas le plus grand score", "49b9743e6f0191561f6769e78002630edacac1d104637b71572295b09cb99bbd": "compare 0,55 et 0,52", "6f824dadd7ebb5a283ffb749b997e9f39ddfcec0cb7d38769e3ec7e3da833dbb": "on prédit la classe de plus GRAND score"}}
📝 Ex 7.Q6b : réponse enregistrée (str).
WB_ANSWER {"hash": "ec96e3cd5b3362766dacf2457883dda6b8c412bc094d7b9036c3bd434cf3f074", "id": "7.Q6c", "kind": "bool", "mistakes": {"b8bc62f0894016d7593e83481394ae3e9de613db63ae03982e9ddc56e3

**7.Q7 — Un-contre-un : duels, votes et coût**

In [9]:
wb.record("7.Q7a", n_ovo(6), mistakes={"A-B et B-A sont le même duel : divise par 2": 30,
          "un-contre-un : un duel par PAIRE de classes, pas un par classe": 6,
          "K × K compte aussi les duels d'une classe contre elle-même, et chaque paire deux fois": 36,
          "c'est le nombre de duels pour 5 classes": 10})
wb.record("7.Q7b", False, mistakes={"un duel ne voit que les échantillons de ses deux classes : les autres sont ignorés": True})
wb.record("7.Q7c", True, mistakes={"calcule 3 × 2 / 2 et compare à 3": False})
wb.record("7.Q7d", "A", mistakes={"A et B sont à égalité : la règle de mylearn garde la classe de plus petit indice": "B",
          "C n'a qu'une voix": "C", "D n'a qu'une voix": "D"})
wb.record("7.Q7e", False, mistakes={"développe K(K − 1)/2 et compare-le à K²/2": True})

WB_ANSWER {"hash": "fb1fa851ada2f896161127be687ac61b54b44c88a5feac2f9cbd2b2d874fbc03", "id": "7.Q7a", "kind": "int", "magnitude": 1, "mistakes": {"4b41d8b820af325e322ece472d01222b5db9b54119f9e0ca1636a4e487af7e01": "un-contre-un : un duel par PAIRE de classes, pas un par classe", "6942939a625631b3af0549ff0d1cbc0b2f9400b44ee52b4fb9d9e8ec4e4876cc": "A-B et B-A sont le même duel : divise par 2", "9915865760b0bd09fd8ef537ef45331f512a61eb2ab94f63783f1eada845974c": "K × K compte aussi les duels d'une classe contre elle-même, et chaque paire deux fois", "e09e0a7f56b2933d5440c46798c1e48842fb6b1800ffa35f0fe1c29e1c48825c": "c'est le nombre de duels pour 5 classes"}, "sign": 1}
📝 Ex 7.Q7a : réponse enregistrée (int).
WB_ANSWER {"hash": "4ae1d4b3250cd483b8f4cb01325778f8f55fd3d0db31c9640a6e81893eed91e3", "id": "7.Q7b", "kind": "bool", "mistakes": {"5722d2af0e0b6a1503f4b92c40e57f508670cdcdda3b3a0e7d61d6c63dea96b1": "un duel ne voit que les échantillons de ses deux classes : les autres sont ignorés"}}

**7.Q8 — Ce que k-means ne peut pas deviner tout seul**

In [10]:
wb.record("7.Q8a", False, mistakes={"k-means est non supervisé : il ne voit que les points": True})
wb.record("7.Q8b", "B", mistakes={"k est fixé AVANT l'entraînement : il n'est pas appris": "A",
          "k est un nombre de clusters, pas une classe": "C"})
wb.record("7.Q8c", False, mistakes={"k-means rend exactement k clusters": True})
wb.record("7.Q8d", False, mistakes={"le résultat dépend des centres de départ : relis « minimum local »": True})
wb.record("7.Q8e", False, mistakes={"que vaut l'inertie quand chaque point est seul dans son cluster ?": True})

WB_ANSWER {"hash": "d8bcb3cb1e722cc30a2473e411ba29f53a4b3aad03d4032dd80da76663b7e81d", "id": "7.Q8a", "kind": "bool", "mistakes": {"efddb26a4148f37a3ee41e5ffd6af84c6b7c6bdb692ab94f572b715e35b6e97b": "k-means est non supervisé : il ne voit que les points"}}
📝 Ex 7.Q8a : réponse enregistrée (bool).
WB_ANSWER {"hash": "77295ebaae00b5271405763d3f4afa4bb02933431a3e7d19b97fd5cf15bdc01f", "id": "7.Q8b", "kind": "str", "mistakes": {"b112e972059e4db32525c2df3b142f9f8592a3eef0e38bc72c285e0788c3fbc8": "k est un nombre de clusters, pas une classe", "e71ff859342531558b746cc2e00a23079abe186f58e1615b5b64ed1a202d1443": "k est fixé AVANT l'entraînement : il n'est pas appris"}}
📝 Ex 7.Q8b : réponse enregistrée (str).
WB_ANSWER {"hash": "79aadf13d9029fc1ef8dac91e57338d8a086fcd5f57660ee8a877c571fa4c340", "id": "7.Q8c", "kind": "bool", "mistakes": {"2190c40e30503da033b261b6c046c6f0299d864f9b674dfe6f4cb4050d629280": "k-means rend exactement k clusters"}}
📝 Ex 7.Q8c : réponse enregistrée (bool).
WB_ANSWER {"

**7.Q9 — Densité d'échantillons quand les features s'accumulent**

In [11]:
wb.record("7.Q9a", 20 // 4, fractional="divise le nombre d'échantillons par le nombre de cases")
wb.record("7.Q9b", 20 / 4 ** 2, decimals=2, mistakes={"en dimension 2, il y a 4 × 4 cases, pas 4 × 2": 20 / 8})
wb.record("7.Q9c", 20 / 4 ** 3, decimals=4, mistakes={"en dimension 3, il y a 4³ = 64 cases, pas 4 × 3": 20 / 12,
          "il y a 4 cases par axe : 4³ cases, pas 3⁴": 20 / 3 ** 4,
          "arrondi trop tôt, ou trop peu de décimales : l'énoncé en demande 4": 0.31})
wb.record("7.Q9d", True, mistakes={"la densité est le nombre d'échantillons divisé par le nombre de cases": False})
wb.record("7.Q9e", True, mistakes={"chaque feature multiplie le nombre de cases par le nombre de cases par axe": False})

WB_ANSWER {"fractional": "divise le nombre d'échantillons par le nombre de cases", "hash": "c3c73f2192ecc7ad635472d45d54ec2f3302d2558d2f499e29407a33a9711dbb", "id": "7.Q9a", "kind": "int", "magnitude": 0, "sign": 1}
📝 Ex 7.Q9a : réponse enregistrée (int).
WB_ANSWER {"alt_coarse": ["156f1d559e7959761f313b106dacd0a6df9d39e56308361ea9a572a7f15df9da"], "decimals": 2, "hash": "312119563f8dd14c616920085c1c32c725a97fd5cf863a0d50d4ef08364b03c9", "hash_coarse": "b94d9e99e73b8a40246fec2a6465a2ad598c03335cd0c12845fdf79c935d2c20", "id": "7.Q9b", "kind": "float", "magnitude": 0, "mistakes": {"b62f4a440d6aa0ec98f7ff01b6080d89418dd5ef56dfee6780479ae222f02462": "en dimension 2, il y a 4 × 4 cases, pas 4 × 2"}, "sign": 1}
📝 Ex 7.Q9b : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"alt_coarse": ["d7a85df4ca4286609a178789066b93023e4336eba0eb385223ae48dd73d1dfd5"], "decimals": 4, "hash": "27b3c3dcd1586494b474d6cb667e4a05c8982d34762f1f8d172c4f9586c2bb90", "hash_coarse": "c58c825e80b46d5668f25cf086

**7.Q11 — Géométrie déroutante en grande dimension**

In [12]:
wb.record("7.Q11a", 1 - 0.97 ** 100 > 0.99, mistakes={"calcule 1 − 0,97¹⁰⁰ et compare-le à 0,99": True})
wb.record("7.Q11b", math.sqrt(10) - 1 > 2, mistakes={"relis les valeurs du livre, ou calcule √10 − 1": False})
wb.record("7.Q11c", True, mistakes={"le coin a toutes ses coordonnées égales à ±1/2 : calcule sa distance au centre en dimension d": False})
wb.record("7.Q11d", "B", mistakes={"en grande dimension, le plus proche voisin s'éloigne vite : relis la concentration des distances": "A",
          "la distance au plus proche voisin ne diminue pas quand la dimension grandit": "C"})

WB_ANSWER {"hash": "7675da64dac18961eb0c4084679bd4928b1765fd87211030824cfdbc2c662b3f", "id": "7.Q11a", "kind": "bool", "mistakes": {"6bc91948e127ebc8091e7827d2060772590893f550da1ff8193371f36efc1eb8": "calcule 1 − 0,97¹⁰⁰ et compare-le à 0,99"}}
📝 Ex 7.Q11a : réponse enregistrée (bool).
WB_ANSWER {"hash": "bcbf9f7418d37c63fc87421bf0f74e4fd1ecc97d1cb092ab20040e6b37626d00", "id": "7.Q11b", "kind": "bool", "mistakes": {"a08d52094ea6363189fcf6a7e630e6496bf07793da7b2bc0488e7df3cf9c81ee": "relis les valeurs du livre, ou calcule √10 − 1"}}
📝 Ex 7.Q11b : réponse enregistrée (bool).
WB_ANSWER {"hash": "58a29bc542c4f425112d49cb4bd7637d7eade180ffb1d02b8a181ae178a750eb", "id": "7.Q11c", "kind": "bool", "mistakes": {"4bdcce7775dc218212639c344bc28e8b20ae834c3391b56eb4217c1558f3409a": "le coin a toutes ses coordonnées égales à ±1/2 : calcule sa distance au centre en dimension d"}}
📝 Ex 7.Q11c : réponse enregistrée (bool).
WB_ANSWER {"hash": "b7462d675e563056d37be6f3705f9360be14666ed279d7bb66acbc6d03ad

**7.R1 — Ch. 6 : entropie d'un cluster pur et d'un cluster mélangé**

In [13]:
wb.record("7.R1a", round(entropy_bits([30])), mistakes={"une seule espèce : il n'y a aucune incertitude": 1})
wb.record("7.R1b", round(entropy_bits([20, 20])), fractional="l'entropie se mesure ici en bits (log2, pas ln), avec un terme par espèce")
wb.record("7.R1c", entropy_bits([0.5, 0.25, 0.25]), decimals=2, mistakes={"en bits, avec log2 (tu as obtenu des nats, avec ln)": 1.5 * math.log(2)})
wb.record("7.R1d", entropy_bits([0.7, 0.2, 0.1]), decimals=3, mistakes={"en bits, avec log2 (tu as obtenu des nats, avec ln)": entropy_bits([0.7, 0.2, 0.1]) * math.log(2),
          "n'oublie pas le signe moins devant la somme": -entropy_bits([0.7, 0.2, 0.1])})
wb.record("7.R1e", math.log2(3), decimals=3, mistakes={"en bits, avec log2 (tu as obtenu des nats, avec ln)": math.log(3),
          "avec trois espèces, la distribution uniforme est [1/3, 1/3, 1/3]": 1.0})

WB_ANSWER {"hash": "4efb43682b7fc5dc0684d501a87dbf224fbb36f08f5126c8d8432842334194a5", "id": "7.R1a", "kind": "int", "magnitude": null, "mistakes": {"2e0753299df316885db9b60ef8c7efdd2ef1cf1593c62a1c19342a2ddf10aad8": "une seule espèce : il n'y a aucune incertitude"}, "sign": 0}
📝 Ex 7.R1a : réponse enregistrée (int).
WB_ANSWER {"fractional": "l'entropie se mesure ici en bits (log2, pas ln), avec un terme par espèce", "hash": "8cd181688961b3a55e794683e7ca464ab7762f732b14d76dac0c1b17edcf1e5d", "id": "7.R1b", "kind": "int", "magnitude": 0, "sign": 1}
📝 Ex 7.R1b : réponse enregistrée (int).
WB_ANSWER {"decimals": 2, "hash": "20ec0874b32a3b3fcbce616b29aeaba8006879cf940b2ac9f7e2ea09cf130d0b", "hash_coarse": "d01570ff72c6b86440203df7f38bb39da880137819ebe60de625012285cdcf45", "id": "7.R1c", "kind": "float", "magnitude": 0, "mistakes": {"269db4b391624101afd2da9cd27cfdd0c9c5e2be8b5109da1ab1de1c70bfcee9": "en bits, avec log2 (tu as obtenu des nats, avec ln)"}, "sign": 1}
📝 Ex 7.R1c : réponse enre

**7.R2 — Ch. 4 : probabilité a posteriori « fécondé » par la règle de Bayes**

In [14]:
wb.record("7.R2a", posterior(0.3, 0.08, 0.04), decimals=3, mistakes={"tu as oublié l'a priori π = 0,3 : multiplie chaque densité par la probabilité de sa classe": posterior(0.5, 0.08, 0.04),
          "c'est le numérateur seul : divise par la somme des deux termes": 0.3 * 0.08,
          "c'est la probabilité de NE PAS être fécondé": 1 - posterior(0.3, 0.08, 0.04)})
wb.record("7.R2b", posterior(0.3, 0.08, 0.04) >= 0.5, mistakes={"compare ta réponse de a) au seuil de 0,5": True})
wb.record("7.R2c", posterior(0.5, 0.08, 0.04), decimals=3, mistakes={"c'est le résultat avec l'a priori de 30 % : refais le calcul avec 0,5 et 0,5": posterior(0.3, 0.08, 0.04)})
wb.record("7.R2d", 0.7 / 0.3, decimals=3, mistakes={"c'est π / (1 − π) : pour une classe RARE, le rapport f1/f0 doit compenser l'a priori défavorable": 0.3 / 0.7,
          "un rapport de 1 ne donne P = 0,5 que si les deux classes sont aussi fréquentes : tiens compte de l'a priori": 1})

WB_ANSWER {"decimals": 3, "hash": "f53d82cf7731873176ee054046b841653d771e41e9b47c1f159625217553ef15", "hash_coarse": "2566fef2ee0c05f12979af55fbdd59cc7c5768d42e80333c5473871f64f71f0f", "id": "7.R2a", "kind": "float", "magnitude": -1, "mistakes": {"3d0b5b4a0166d9d8a57397a1d17ce3ee9cfac9d466667638e42ca448c8dfbcd7": "tu as oublié l'a priori π = 0,3 : multiplie chaque densité par la probabilité de sa classe", "849e82799f94e9c56c0c50a58ac7042ed7106b6ab9a41f85a50bc798519b6aa9": "c'est le numérateur seul : divise par la somme des deux termes", "fbad87117fa6b3f6a71a41945ed66f45929f2e773c2e96d2909d4ed5e2dafb32": "c'est la probabilité de NE PAS être fécondé"}, "sign": 1}
📝 Ex 7.R2a : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"hash": "204b386f07a0ffc36f73e1c0a7a3926b61a04586e2ced985c95e23efdafca9f9", "id": "7.R2b", "kind": "bool", "mistakes": {"75bf55d1d089a0199ddecd21cce8cb44a637e1eb9ac7c14c8b0f5ad3ee4d8b1b": "compare ta réponse de a) au seuil de 0,5"}}
📝 Ex 7.R2b : réponse enregist

**7.R3 — 0B : développer ‖a − b‖² avec le produit scalaire**

In [15]:
wb.record("7.R3a", int(A_R3 @ A_R3), fractional="‖a‖² est le CARRÉ de la norme : la somme des carrés des coordonnées, sans racine",
          mistakes={"‖a‖² est la somme des CARRÉS des coordonnées": 4})
wb.record("7.R3b", int(A_R3 @ B_R3), mistakes={"a · b = a1 b1 + a2 b2 + a3 b3 : attention au signe de la deuxième coordonnée de a": 5})
wb.record("7.R3c", int(B_R3 @ B_R3), fractional="‖b‖² est le CARRÉ de la norme : la somme des carrés des coordonnées, sans racine",
          mistakes={"‖b‖² est la somme des CARRÉS des coordonnées": 3})
wb.record("7.R3d", int((A_R3 - B_R3) @ (A_R3 - B_R3)), fractional="on demande le CARRÉ de la distance : la somme des carrés des coordonnées de a − b, sans racine",
          mistakes={"le terme croisé vaut −2 a·b, pas −a·b": 18, "tu as oublié le terme croisé −2 a·b": 19,
          "le terme croisé est MOINS 2 a·b": 21})

WB_ANSWER {"fractional": "‖a‖² est le CARRÉ de la norme : la somme des carrés des coordonnées, sans racine", "hash": "9d238c5f73856945247db10355d8ae9da53cde4316235f782dbe81e0c6218f76", "id": "7.R3a", "kind": "int", "magnitude": 1, "mistakes": {"d5176b15ffe1aee4dff674c4f005d78ded73cf9f19773a05bfee2c411c91548c": "‖a‖² est la somme des CARRÉS des coordonnées"}, "sign": 1}
📝 Ex 7.R3a : réponse enregistrée (int).
WB_ANSWER {"hash": "21d178dd598f00f83a9270dba43db5e667773797c57956a4d71f2c91862b3851", "id": "7.R3b", "kind": "int", "magnitude": 0, "mistakes": {"b9ae4435d2cd656e939384f7b35a6c3b0d1705bff6e0b2097aca210634080741": "a · b = a1 b1 + a2 b2 + a3 b3 : attention au signe de la deuxième coordonnée de a"}, "sign": 1}
📝 Ex 7.R3b : réponse enregistrée (int).
WB_ANSWER {"fractional": "‖b‖² est le CARRÉ de la norme : la somme des carrés des coordonnées, sans racine", "hash": "9774742e3883c460aa72c92a78871effe1fbc6baee45d53624f2e96114b51cac", "id": "7.R3c", "kind": "int", "magnitude": 0, "mista

**Ex 7.1 — Compter les classifieurs OvR et OvO**

In [16]:
wb.record("7.1a", [3, n_ovo(3)], mistakes={"A-B et B-A sont le même duel : divise K(K − 1) par 2": [3, 6]})
wb.record("7.1b", [10, n_ovo(10)], mistakes={"A-B et B-A sont le même duel : divise K(K − 1) par 2": [10, 90],
          "l'ordre demandé est [N_OvR, N_OvO]": [45, 10],
          "une classe ne se bat pas contre elle-même, et chaque paire ne compte qu'une fois": [10, 100]})
wb.record("7.1c", [26, n_ovo(26)], mistakes={"A-B et B-A sont le même duel : divise K(K − 1) par 2": [26, 650],
          "l'ordre demandé est [N_OvR, N_OvO]": [325, 26]})
wb.record("7.1d", [100, n_ovo(100)], mistakes={"A-B et B-A sont le même duel : divise K(K − 1) par 2": [100, 9900],
          "l'ordre demandé est [N_OvR, N_OvO]": [4950, 100],
          "K²/2 compte aussi les duels d'une classe contre elle-même : c'est K(K − 1)/2": [100, 5000]})
wb.record("7.1e", next(k for k in range(2, 1000) if n_ovo(k) > 10_000), fractional="cherche le plus petit ENTIER K tel que K(K − 1)/2 dépasse 10 000",
          mistakes={"avec 141 classes, il n'y a encore que 9 870 duels : il en faut PLUS de 10 000": 141,
          "n'oublie pas de diviser par 2 : il faut K(K − 1)/2 > 10 000, c'est-à-dire K(K − 1) > 20 000": 101})
wb.record("7.1f", 10 * 50_000, mistakes={"chaque classifieur voit TOUTES les images, et il y a 10 classifieurs": 50_000,
          "chaque classifieur de l'un-contre-tous voit TOUTES les images, pas seulement celles de deux chiffres": 45 * 10_000})
wb.record("7.1g", n_ovo(10) * 2 * 5_000, mistakes={"un duel ne voit que les images de ses DEUX chiffres (10 000), pas toutes": 45 * 50_000,
          "un duel ne voit que les images de ses deux chiffres : compte les duels, puis les images de chacun": 10 * 50_000,
          "chaque duel voit les images de ses DEUX chiffres : 2 × 5 000": 45 * 5_000,
          "A-B et B-A sont le même duel : il y a K(K − 1)/2 duels, pas K(K − 1)": 90 * 10_000})
wb.record("7.1h", (10 * 50_000 ** 2) / (n_ovo(10) * 10_000 ** 2), decimals=1, mistakes={"c'est le rapport inverse : divise le coût de l'un-contre-tous par celui de l'un-contre-un": (n_ovo(10) * 10_000 ** 2) / (10 * 50_000 ** 2),
          "le coût d'un modèle est m² : élève au carré le nombre d'exemples de CHAQUE modèle avant d'additionner": (10 * 50_000) / (n_ovo(10) * 10_000)})

WB_ANSWER {"decimals": 0, "element_hashes": ["a43db5f0b6ea562cc6e2e84aca4313e80c932139ae8f67ffbdc96446d95c5964", "1481379faafd96ae112495f43aada42eeeeedb6581c6b80140bf2d17af3c3571"], "hash": "76e0c9d0be8f81b3761c37bd2643004d6570a3bbf3a37c518b418ffa7b4e1428", "id": "7.1a", "integer": true, "kind": "array", "mistakes": {"7edf908edb39d31fb72fbde3c7e54ecd0e5f508e9f3fa5fb581c167716c059eb": "A-B et B-A sont le même duel : divise K(K − 1) par 2"}, "shape": [2]}
📝 Ex 7.1a : réponse enregistrée (array, 0 décimale(s)).
WB_ANSWER {"decimals": 0, "element_hashes": ["defee9aebd0ac54a780278bf8837b73fb5b3c12cbfa9e4952f27322eba697d08", "ab325613c4876bb9b8b37d5c41894d56cfa1bbcf52793cafad44eb41015cb125"], "hash": "01c62b41ff33d3fd2bf015f269aef66b68789d7db57c96c880827619f5fb4053", "id": "7.1b", "integer": true, "kind": "array", "mistakes": {"1368887bd75fa3222737799d7a6458bdea2fc4332ab846999e3ced18643ca525": "A-B et B-A sont le même duel : divise K(K − 1) par 2", "1f920e958c18bb951e4dd5495dff7ff7cd2d00ca23

**Ex 7.2 — Dépouiller les votes d'un un-contre-un à quatre classes**

In [17]:
wb.record("7.2a", votes(WINNERS_72A).tolist(), mistakes={"tu as compté les défaites : une voix par duel GAGNÉ": (3 - votes(WINNERS_72A)).tolist()})
wb.record("7.2b", "B", mistakes={"compte les voix de a) : D n'en a pas le plus": "D", "A ne gagne qu'un duel": "A",
          "C ne gagne aucun duel": "C"})
wb.record("7.2c", int(votes(WINNERS_72A).sum()), mistakes={"chaque duel donne UNE voix, à son vainqueur seulement": 12})
wb.record("7.2d", votes(WINNERS_72B).tolist(), mistakes={"tu as compté les défaites : une voix par duel GAGNÉ": (3 - votes(WINNERS_72B)).tolist()})
wb.record("7.2e", "A", mistakes={"A et C sont à égalité : la règle de mylearn garde la classe de plus petit INDICE": "C",
          "B n'a qu'une voix": "B", "D n'a qu'une voix": "D"})
wb.record("7.2f", "C", mistakes={"regarde qui a gagné le duel A-C": "A", "B n'a qu'une voix": "B", "D n'a qu'une voix": "D"})
wb.record("7.2g", False, mistakes={"une classe qui gagne ses K − 1 duels a K − 1 voix ; combien au plus pour chacune des autres, qui ont toutes perdu contre elle ?": True})
wb.record("7.2h", False, mistakes={"6 voix pour 4 classes : combien chacune en aurait-elle en cas d'égalité parfaite ?": True})

WB_ANSWER {"decimals": 0, "element_hashes": ["8134a2bc4355003cf267f92cf51bc39547900c4d5a48e3b12ec514f1b1d54f0e", "5288910204d5d8d0d19bef38ff58c13527e9f7f0ba686eca3e8dab2486b5d8e7", "d646624910401ca495f110a98df6c9981ae3a58689789bd17ef9f3db28ed331f", "78e23d9aebc80421f7bef0f776f2bd7e66c681dc558daa9b555ba7a17491ea38"], "hash": "e8be76ec0859fda512ba9fb0c7185a7e7192f836d7e859d44f493c89ef62c13e", "id": "7.2a", "integer": true, "kind": "array", "mistakes": {"9d4d0587af8fbf4fb5fa647aaf605c94da0180d00acffe0f6f04b76e9fc3566a": "tu as compté les défaites : une voix par duel GAGNÉ"}, "shape": [4]}
📝 Ex 7.2a : réponse enregistrée (array, 0 décimale(s)).
WB_ANSWER {"hash": "7df79092ec6f34dc4022724fe826bd2eb50cbc22f4fc722f9580aedb3708f7c0", "id": "7.2b", "kind": "str", "mistakes": {"4ffb580d8c7ed2f9e20f341b6ce66da098086ca1a59c3ffd897949c209800478": "compte les voix de a) : D n'en a pas le plus", "789e1a33d5281b34172a28a31d428f24a59a2ccd4e7f92b76f15220b3ff0ff50": "A ne gagne qu'un duel", "ddab88569a7a

**Ex 7.3 — Une itération de k-means à la main**

In [18]:
wb.record("7.3a", (L1_73 + 1).tolist(), mistakes={"numérote les clusters 1 et 2, comme l'énoncé": L1_73.tolist(),
          "P2 est à 1 de c1 et à 5 de c2 (distances au carré)": [1, 2, 2, 2, 2, 2],
          "numérote les clusters comme l'énoncé : le cluster 1 est celui du centre de départ c1 = P1": (2 - L1_73).tolist()})
wb.record("7.3b", round(inertia(P_73, C0_73, L1_73)), fractional="l'inertie de b) se calcule avec les centres de DÉPART (P1 et P3) et des distances AU CARRÉ : avec ces coordonnées entières, elle est entière",
          mistakes={"l'inertie est une SOMME, pas une moyenne": round(inertia(P_73, C0_73, L1_73) / 6)})
wb.record("7.3c", C1_73.tolist(), decimals=2, mistakes={"c2 est la moyenne de P3, P4, P5 ET P6 : P3 est encore dans le cluster 2": [[1.5, 1.0], [6.0, 13 / 3]],
          "dans l'ordre [c1, c2] : c1 est le centre du cluster de P1": C1_73[::-1].tolist()})
wb.record("7.3d", inertia(P_73, C1_73, L1_73), decimals=2, mistakes={"garde l'affectation de a) : P3 est encore dans le cluster 2 à cette étape": inertia(P_73, C1_73, L2_73)})
wb.record("7.3e", int(np.flatnonzero(L2_73 != L1_73)[0]) + 1, mistakes={"recalcule les distances de P4 aux deux centres de c) : il reste bien plus près de c2": 4})
wb.record("7.3f", C2_73.tolist(), decimals=3, mistakes={"ce sont les centres de c) : refais la mise à jour avec la nouvelle affectation": C1_73.tolist(),
          "dans l'ordre [c1, c2] : c1 est le centre du cluster de P1": C2_73[::-1].tolist()})
wb.record("7.3g", inertia(P_73, C2_73, L2_73), decimals=2, mistakes={"c'est l'inertie de d) : recalcule-la avec les centres de f)": inertia(P_73, C1_73, L1_73),
          "c'est l'inertie avant la dernière mise à jour : recalcule-la avec les centres de f)": inertia(P_73, C1_73, L2_73)})
wb.record("7.3h", bool((L3_73 != L2_73).any()), mistakes={"recalcule les distances de P3 et de P4 aux centres de f) : changent-ils encore de cluster ?": True})
wb.record("7.3i", True, mistakes={"relis les inerties de b), d) et g) : ont-elles augmenté ?": False})

WB_ANSWER {"decimals": 0, "element_hashes": ["bdc9fdf84894b8e88e0deca7550f8b7bad1d6c91eb62ee43a72afe04a24fc462", "3b656aa8071ad6605e3f4f6cafae5feb3573110b070c55c3b7067cabf5607bac", "fba299d6cdb2a92a9111fca5b786a2514cf762f95b06519bbafda711359998d0", "0251284671519ba015c343c2840ed5379c0b75dfbd8f3e47b0f0bb2c32873a96", "be1d9af98494559cb417c1913190f714d393582c60fd2fccb3ac47ed7094f2cd", "82ae1144593c72c8e092fbc858f2b0f1b1d5a51c354c82fb1f69521c1ae250d3"], "hash": "fef85e41b8deb59a7433408df1e92590b6bdffef39b5db96d7df907100983b51", "id": "7.3a", "integer": true, "kind": "array", "mistakes": {"41fb2a3c1d9828d13e9d0ce659de823c79cf198140eba628e2f215c10152071d": "numérote les clusters comme l'énoncé : le cluster 1 est celui du centre de départ c1 = P1", "dcf2714ed2afee30d71882ddfb6ed46f2af57ffd69f4bab42fe25126cf7bd2da": "numérote les clusters 1 et 2, comme l'énoncé", "df2ecad49d81fd6aec4659211f75dfa7d15fd0f7ed381750cc66226f96a5777d": "P2 est à 1 de c1 et à 5 de c2 (distances au carré)"}, "shape": 

**Ex 7.4 — Densité d'échantillons et nombre d'œufs nécessaires**

In [19]:
wb.record("7.4a", [N_74 / B_74 ** d for d in range(1, 5)], decimals=3, mistakes={"en dimension d, il y a 6^d cases, pas 6 × d": [N_74 / (B_74 * d) for d in range(1, 5)]})
wb.record("7.4b", 2 * B_74 ** 3, mistakes={"n = ρ × b^d, pas ρ × b × d": 2 * B_74 * 3, "multiplie le nombre de cases par la densité 2": B_74 ** 3})
wb.record("7.4c", 2 * B_74 ** 6, mistakes={"multiplie le nombre de cases par la densité 2": B_74 ** 6, "n = ρ × b^d, pas ρ × b × d": 2 * B_74 * 6})
wb.record("7.4d", 2 * 10 ** 6, mistakes={"multiplie le nombre de cases par la densité 2": 10 ** 6})
wb.record("7.4e", next(d for d in range(1, 30) if 10 ** 6 / B_74 ** d < 1), fractional="une dimension est un nombre entier : prends le plus petit entier d tel que 10⁶ / 6^d < 1",
          mistakes={"en dimension 7, la densité vaut encore 10⁶ / 6⁷ ≈ 3,6 : elle n'est pas sous 1": 7})
wb.record("7.4f", EMPTY_74, decimals=3, mistakes={"c'est la probabilité qu'il soit OCCUPÉ : on demande celle qu'il soit vide": 1 - EMPTY_74,
          "il y a 10 œufs indépendants : chacun doit éviter le cube": 124 / 125,
          "1 − 0,08 ? la densité n'est pas une probabilité : chaque œuf doit éviter ce cube, indépendamment des autres (si tu as seulement arrondi, donne 3 décimales)": 1 - 10 / 125})
wb.record("7.4g", 125 * (1 - EMPTY_74), decimals=2, mistakes={"deux œufs peuvent tomber dans le même cube : multiplie 125 par la probabilité qu'un cube soit occupé": 10.0,
          "c'est le nombre moyen de cubes VIDES": 125 * EMPTY_74,
          "tu as utilisé la valeur arrondie de f) : garde sa valeur exacte jusqu'au bout": 9.62,
          "tu as arrondi f) avant de t'en servir : garde toutes ses décimales jusqu'à la fin du calcul": 9.63})

WB_ANSWER {"decimals": 3, "element_coarse": [["0c6e1a42e7bf3596d4547474cd6342337a44d13d9326b0e934de7ba5f16de9d2"], ["b397fa142184fce55764204f8739ab2d8c761bc64f33698eea68ce3dbb3b93de"], ["e22f73f7898b7a1d43b393a9aec93739188a8be10ab816952a09054f60847756"], ["1fffbcc9fce82e290ac5f66bf43a378fd77dbbf1c7f75cc260c69cfcfc9e9956"]], "element_hashes": ["21873f784bb31f167ba78e6ee9a06e3f027336bc3af58ddb38163245bc986d9a", "d9bb416ad6fd87d5891bc0031bf7fc4c1240a26500c2fba4fb14265bc1f84a3b", "4478611144b4af6bc2f6d87149755d136fc93389cfc7285e8b4cd7832a5e17cc", "75fbd3774ce15e196716ce4d18be437ab5c5f5fe450cc6cd9254bbbbfa60465b"], "hash": "558d7f645a0c1a03452605d37ab16f852eb74b85716a12527b7ab6e60d529254", "id": "7.4a", "kind": "array", "mistakes": {"3cbbc2e3daf54ce7d2abfebee6a74d027747ab28fe7058f0ce031e96a1df55c7": "en dimension d, il y a 6^d cases, pas 6 × d"}, "shape": [4]}
📝 Ex 7.4a : réponse enregistrée (array, 3 décimale(s)).
WB_ANSWER {"hash": "694328edcef44925d2c8827e51816b30bd5f8956b76cdfd52ad5c672

**Ex 7.5 — Le rayon de l'hyper-orange : r(d) = √d − 1**

In [20]:
wb.record("7.5a", math.sqrt(2) - 1, decimals=3, mistakes={"le rayon de l'orange est √d − 1 : il faut retirer le rayon d'un ballon": math.sqrt(2),
          "c'est le diamètre de l'orange : on demande son rayon": 2 * (math.sqrt(2) - 1)})
wb.record("7.5b", math.sqrt(3) - 1, decimals=3, mistakes={"le rayon de l'orange est √d − 1 : il faut retirer le rayon d'un ballon": math.sqrt(3),
          "c'est le diamètre de l'orange : on demande son rayon": 2 * (math.sqrt(3) - 1)})
wb.record("7.5c", 2 ** 12, mistakes={"un ballon par coin : chaque coordonnée d'un coin vaut −1 ou +1, indépendamment des autres : combien de choix en tout ?": 2 * 12,
          "12² ne compte que des paires : chaque coordonnée d'un coin vaut −1 ou +1, indépendamment des autres : combien de choix en tout ?": 12 ** 2})
wb.record("7.5d", next(d for d in range(1, 100) if math.sqrt(d) - 1 >= 1.5), fractional="une dimension est un nombre entier : prends le plus petit entier qui convient",
          mistakes={"en dimension 6, r = √6 − 1 ≈ 1,45 : pas encore 1,5": 6,
          "r = √d − 1 : n'oublie pas de retirer le rayon d'un ballon": 3})
wb.record("7.5e", next(d for d in range(1, 100) if math.sqrt(d) - 1 > 2.5), fractional="une dimension est un nombre entier : prends le plus petit entier qui convient",
          mistakes={"en dimension 12, r = √12 − 1 ≈ 2,46 : pas encore 2,5": 12,
          "r = √d − 1 : n'oublie pas de retirer le rayon d'un ballon": 7,
          "c'est le seuil r > 2 du livre (l'orange sort de la boîte) : ici, on demande r > 2,5": 10})
wb.record("7.5f", math.sqrt(50) - 1, decimals=3, mistakes={"le rayon de l'orange est √d − 1 : il faut retirer le rayon d'un ballon": math.sqrt(50),
          "c'est le diamètre de l'orange : on demande son rayon": 2 * (math.sqrt(50) - 1)})

WB_ANSWER {"decimals": 3, "hash": "ec9cfc181b1856b59fc57d6635e84264c4b5d32ed199cff12635b2fd5ac57d9c", "hash_coarse": "bfbb2aeb21d5d39f6c0b0dcfeee37cf4aae6d9f83f048ad1f9ad40edb707285b", "id": "7.5a", "kind": "float", "magnitude": -1, "mistakes": {"43ce692f83f196540101171baab3e9e5543e5edde8b9469f363f577a18ec255b": "le rayon de l'orange est √d − 1 : il faut retirer le rayon d'un ballon", "846c3156afbbbe462601401ba8a986cf5d9944046fcafde693438ae93eab620f": "c'est le diamètre de l'orange : on demande son rayon"}, "sign": 1}
📝 Ex 7.5a : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "134a5ac116dfa12fbd80d778d8892975d60d141d596a0cd735ffd870b60dc96b", "hash_coarse": "e503228bcdc271a4439e29193fb178735720d3303b6e333363c35cc5091a7e08", "id": "7.5b", "kind": "float", "magnitude": -1, "mistakes": {"005536c878f919394272afb05155692b25d24c177abdca91ea12ffe049ab0d09": "c'est le diamètre de l'orange : on demande son rayon", "e2a73eb761d91a980f93a1d4f731c6f98fc395b10e7b3af5e

**Ex 7.6 — Boule dans un cube : rapport des volumes par récurrence**

In [21]:
wb.record("7.6a", V[4], decimals=3, mistakes={"la récurrence saute de deux dimensions : V4 = (2π/4) V2, pas (2π/4) V3": 2 * math.pi / 4 * V[3]})
wb.record("7.6b", V[5], decimals=3, mistakes={"la récurrence saute de deux dimensions : V5 = (2π/5) V3, pas (2π/5) V4": 2 * math.pi / 5 * V[4]})
wb.record("7.6c", V[4] / 2 ** 4, decimals=3, mistakes={"le cube de côté 2 a pour volume 2⁴ = 16 en dimension 4": V[4] / 4,
          "le cube de côté 2 a pour volume 2^d : ici 2⁴": V[4] / 2 ** 5,
          "la boule de rayon 1 tient dans le cube de côté 2, pas 1 : divise par le volume de ce cube": V[4]})
wb.record("7.6d", 100 * V[10] / 2 ** 10, decimals=3, mistakes={"c'est une proportion : l'énoncé demande un pourcentage, avec 3 décimales": 0.002,
          "multiplie par 100 : l'énoncé demande un pourcentage, pas une proportion": 0.003,
          "la boule de rayon 1 tient dans le cube de côté 2, pas 1 : divise par le volume de ce cube": V[10],
          "le cube qui contient la boule de rayon 1 a pour côté 2 : divise V10 par son volume, puis multiplie par 100": 100 * V[10]})
wb.record("7.6e", max(range(1, 30), key=lambda d: V[d]), mistakes={"tu n'as peut-être calculé que les dimensions paires : la récurrence saute de deux en deux, calcule aussi V3, V5 et V7": 6,
          "tu t'es peut-être arrêté trop tôt : calcule V1 à V8 avant de chercher le maximum": 4})
wb.record("7.6f", 1 - 0.95 ** 100, decimals=3, mistakes={"c'est la part du cœur, l'intérieur de rayon 0,95 : on demande celle de la peau": 0.95 ** 100,
          "le volume est proportionnel à r^d : la peau contient bien plus que 5 % du volume (relis « Tout est dans l'écorce »)": 0.05,
          "c'est la valeur de la fiche pour une peau de 10 % en dimension 50 : ici, la peau fait 5 % et d = 100": 0.995})

WB_ANSWER {"decimals": 3, "hash": "d66c8039fd02e31c089ce16a0db053d85caf3f0030c785baabe071729ec7a118", "hash_coarse": "a7de23364510c6067a055534b992c82a9c63051778f4acc247cff49744538a0f", "id": "7.6a", "kind": "float", "magnitude": 0, "mistakes": {"e69d6a34ed6ad97f454efbb568c6ba33afe9f37b6acb36bcbdaeb9e874eb372f": "la récurrence saute de deux dimensions : V4 = (2π/4) V2, pas (2π/4) V3"}, "sign": 1}
📝 Ex 7.6a : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "770cf144685cd48d927e89875917c1fa9f0be07cc4de618367eeb7bcbdf25bf1", "hash_coarse": "eebb61f9ebafdc1c505a0dc2046644e2db61066be536129f3ec563ab902fbb9c", "id": "7.6b", "kind": "float", "magnitude": 0, "mistakes": {"d7af25d019a1314d37518f1b9d3ddcde2b97eaeb6c4e29a25b7479f6fa819989": "la récurrence saute de deux dimensions : V5 = (2π/5) V3, pas (2π/5) V4"}, "sign": 1}
📝 Ex 7.6b : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "4205893cfaa95357f72f532d8cda2d319f51954260710a83553d5d8

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu choisir un seuil de décision d'après le coût des deux erreurs, et dire qui doit fixer ces coûts ?
2. Sais-tu compter et dépouiller les votes d'un un-contre-un, et dire quand le préférer à l'un-contre-tous ?
3. Sais-tu dérouler k-means à la main, et expliquer pourquoi la densité des exemples s'effondre quand on ajoute des features ?

**Pour aller plus loin** : les guides « Clustering » et « Multiclass and multioutput algorithms » de scikit-learn, cités dans la fiche. La suite : le ch. 8 (entraînement et test), où un jeu de validation fixe les seuils et les hyperparamètres comme $k$.

*Les exercices de code 7.11 à 7.31 (parties A à D, tes librairies `mylearn.cluster` et `mylearn.multiclass`) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py 7` pour obtenir le notebook complet.*